# تشغيل خادم «سعّر» على Kaggle

ارفع `saer-backend-bundle.zip` كـ Kaggle Dataset **خاص** باسم `saer-backend-bundle`، وأضفه إلى هذا الدفتر. فعّل Internet وGPU.

في إعدادات Kaggle > Secrets يمكنك إضافة `NGROK_AUTHTOKEN` و`NGROK_STATIC_DOMAIN`، واختيارياً `GITHUB_GIST_TOKEN` و`GIST_ID`. النطاق المجاني هو النطاق dev الذي يعيّنه ngrok لحسابك (حالياً `ngrok-free.dev`)؛ استخدمه كما هو ولا تختر نطاقاً مخصصاً. يبقى الاسم ثابتاً بعد إعادة تشغيل النفق، لكن الخادم لا يستجيب عند توقف جلسة Kaggle. لا تكتب الرموز داخل الخلايا. عند وجود إعداد ngrok صالح يختاره الدفتر، وإلا يستخدم Gradio Share.

In [ ]:
%pip install -q gradio==5.49.1
print('تم تثبيت Gradio. ستُفحص بقية المتطلبات بعد قراءة الحزمة.')

In [ ]:
from pathlib import Path
import zipfile

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/saer-backend-bundle')
if not INPUT_ROOT.is_dir():
    raise RuntimeError('لم أجد /kaggle/input؛ أضف Dataset الخاص إلى هذا الدفتر أولاً.')
roots = [INPUT_ROOT / 'saer-backend-bundle']
roots += [p for p in INPUT_ROOT.iterdir() if p.is_dir() and p not in roots]
bundle_root = next((p for p in roots if p.is_dir() and (p / 'app.py').is_file() and (p / 'model_store.py').is_file()), None)
if bundle_root is None:
    archives = sorted(INPUT_ROOT.rglob('saer-backend-bundle.zip')) or sorted(INPUT_ROOT.rglob('*.zip'))
    if len(archives) != 1:
        raise RuntimeError(f'توقعت ملف حزمة ZIP واحداً؛ وجدت: {len(archives)}')
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(WORK_ROOT)
    bundle_root = WORK_ROOT
if not (bundle_root / 'requirements.txt').is_file():
    raise RuntimeError(f'requirements.txt غير موجود في {bundle_root}')
print(f'مسار الحزمة: {bundle_root}')
print('عدد الملفات:', sum(1 for p in bundle_root.rglob('*') if p.is_file()))

In [ ]:
import importlib.metadata as metadata, subprocess, sys
from packaging.requirements import Requirement
from packaging.version import Version

required = []
pip_options = []
for raw in (bundle_root / 'requirements.txt').read_text(encoding='utf-8').splitlines():
    raw = raw.strip()
    if not raw or raw.startswith('#'):
        continue
    if raw.startswith("--extra-index-url "):
        pip_options.extend(raw.split())
        continue
    req = Requirement(raw)
    try:
        installed = Version(metadata.version(req.name))
        okay = not req.specifier or req.specifier.contains(installed, prereleases=True)
    except metadata.PackageNotFoundError:
        okay = False
    if not okay:
        required.append(str(req))
if required:
    print('تثبيت الناقص أو غير المطابق:', ', '.join(required))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pip_options, *required], check=True)
else:
    print('المتطلبات المثبتة مطابقة للإصدارات المحددة.')

In [ ]:
# عامل الصورة يستخدم إصدار التدريب دون تبديل بيئة نموذج المواصفات والهوية.
import os, subprocess, sys
from pathlib import Path
image_env = Path('/kaggle/working/saer-image-env')
if not (image_env / 'bin/python').is_file():
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(image_env)], check=True)
image_python = image_env / 'bin/python'
subprocess.run([str(image_python), '-m', 'pip', 'install', '-q', '--no-deps', 'scikit-learn==1.6.1'], check=True)
os.environ['SAER_IMAGE_PYTHON'] = str(image_python)


In [ ]:
import hashlib, json, os, sys

manifest = json.loads((bundle_root / 'private_assets_manifest.json').read_text(encoding='utf-8'))
problems = []
for item in manifest['files']:
    path = bundle_root / item['path']
    if not path.is_file():
        problems.append(f"مفقود: {item['path']}")
        continue
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if path.stat().st_size != item['size_bytes'] or digest != item['sha256']:
        problems.append(f"فشل الحجم أو SHA-256: {item['path']}")
if problems:
    raise RuntimeError('\n'.join(problems))
print(f"فحصت الأصول الستة: {manifest['total_bytes']:,} بايت؛ كل البصمات مطابقة.")

PAGES_PROJECT_URL = 'https://saer-rmsavg637.pages.dev'  # استبدله برابط Pages الفعلي
from urllib.parse import urlsplit
parsed_pages = urlsplit(PAGES_PROJECT_URL)
PAGES_ORIGIN = f'{parsed_pages.scheme}://{parsed_pages.netloc}'
os.environ['SAER_APP_ROOT'] = str(bundle_root)
os.environ['LOCAL_ASSETS_DIR'] = str(bundle_root)
os.environ['SAER_RUNTIME_CACHE'] = '/kaggle/working/saer-runtime-cache'
os.environ['FRONTEND_ORIGINS'] = PAGES_ORIGIN
sys.path.insert(0, str(bundle_root))
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
import json, logging, os, queue, re, subprocess, threading, time, urllib.request
from urllib.parse import quote
import requests

def kaggle_secret(name):
    value = os.environ.get(name, '')
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name) or value
    except Exception:
        pass
    return value.strip() if isinstance(value, str) else ''

NGROK_AUTHTOKEN = kaggle_secret('NGROK_AUTHTOKEN')
NGROK_STATIC_DOMAIN = kaggle_secret('NGROK_STATIC_DOMAIN')
GITHUB_GIST_TOKEN = kaggle_secret('GITHUB_GIST_TOKEN')
GIST_ID = kaggle_secret('GIST_ID')
TUNNEL_MODE = 'auto'  # auto أو ngrok أو gradio أو cloudflared
mode = TUNNEL_MODE
if mode == 'auto':
    mode = 'ngrok' if NGROK_AUTHTOKEN else 'gradio'
import app as saer_server
if saer_server._startup_error:
    raise RuntimeError(saer_server._startup_error)

def start_uvicorn():
    import uvicorn
    threading.Thread(target=lambda: uvicorn.run(saer_server.app, host='0.0.0.0', port=7860, log_level='warning'), daemon=True).start()
    for _ in range(90):
        try:
            urllib.request.urlopen('http://127.0.0.1:7860/config', timeout=2).read(64)
            return
        except Exception:
            time.sleep(1)
    raise RuntimeError('لم يبدأ خادم Uvicorn على المنفذ 7860.')

api_url = None
if mode == 'ngrok':
    if not NGROK_AUTHTOKEN:
        raise RuntimeError('أضف NGROK_AUTHTOKEN إلى Kaggle Secrets أو استخدم TUNNEL_MODE=gradio.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok==9.1.2'], check=True)
    from pyngrok import conf, ngrok
    logging.getLogger('pyngrok').setLevel(logging.WARNING)
    config_path = '/kaggle/temp/saer-ngrok.yml'
    ngrok_config = conf.PyngrokConfig(config_path=config_path)
    ngrok.set_auth_token(NGROK_AUTHTOKEN, pyngrok_config=ngrok_config)
    start_uvicorn()
    clean_domain = NGROK_STATIC_DOMAIN.strip().removeprefix('https://').removeprefix('http://').split('/', 1)[0]
    if clean_domain:
        tunnel = ngrok.connect(addr=7860, proto='http', domain=clean_domain, pyngrok_config=ngrok_config)
    else:
        tunnel = ngrok.connect(addr=7860, proto='http', pyngrok_config=ngrok_config)
    api_url = tunnel.public_url
    try:
        from pathlib import Path
        Path(config_path).unlink(missing_ok=True)
    except Exception:
        pass
elif mode == 'gradio':
    result = saer_server.launch_gradio_share(prevent_thread_lock=True)
    api_url = next((v for v in result if isinstance(v, str) and '.gradio.live' in v), None) if isinstance(result, (tuple, list)) else None
    api_url = api_url or getattr(saer_server.demo, 'share_url', None)
elif mode == 'cloudflared':
    import urllib.request
    start_uvicorn()
    cloudflared = '/kaggle/working/cloudflared'
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    os.chmod(cloudflared, 0o755)
    proc = subprocess.Popen([cloudflared, 'tunnel', '--url', 'http://127.0.0.1:7860', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    threading.Thread(target=lambda: [lines.put(line) for line in proc.stdout], daemon=True).start()
    deadline = time.time() + 90
    while time.time() < deadline and not api_url:
        try:
            line = lines.get(timeout=1)
        except queue.Empty:
            continue
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            api_url = match.group(0)
elif mode != 'gradio':
    raise ValueError('TUNNEL_MODE يجب أن يكون auto أو ngrok أو gradio أو cloudflared.')
if not api_url:
    raise RuntimeError('لم يُستخرج رابط HTTPS عام؛ راجع الإنترنت وإعداد النفق.')
api_url = api_url.rstrip('/')
print('رابط API العام:')
print(api_url)

GIST_FILENAME = 'saer-api.json'
gist_config_url = ''
if GITHUB_GIST_TOKEN and GIST_ID:
    payload = {'api_base_url': api_url, 'updated_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
    headers = {'Authorization': f'Bearer {GITHUB_GIST_TOKEN}', 'Accept': 'application/vnd.github+json', 'X-GitHub-Api-Version': '2022-11-28'}
    response = requests.patch(f'https://api.github.com/gists/{GIST_ID}', headers=headers, json={'files': {GIST_FILENAME: {'content': json.dumps(payload, ensure_ascii=False)}}}, timeout=30)
    if not response.ok:
        raise RuntimeError(f'فشل تحديث Gist (HTTP {response.status_code})؛ لم يُعرض رمز الدخول.')
    gist_config_url = f'https://api.github.com/gists/{GIST_ID}'
    print('تم تحديث ملف الإعداد العام في Gist دون عرض الرمز.')

if gist_config_url:
    frontend_url = f'{PAGES_PROJECT_URL.rstrip("/")}/?apiConfig={quote(gist_config_url, safe="")}'
else:
    frontend_url = f'{PAGES_PROJECT_URL.rstrip("/")}/?api={quote(api_url, safe="")}'
print('\n' + '=' * 72)
print('الرابط النهائي الجاهز لواجهة Cloudflare Pages:')
print(frontend_url)
print('=' * 72)
print('لا توقف جلسة Kaggle؛ يتوقف الخادم عند إنهائها. الرابط ngrok قد يبقى ثابتاً لكنه لن يستجيب أثناء توقف الجلسة.')